# AI Engineer Interview Prep — Sections 8–12
This notebook covers prompting techniques, function calling, fine-tuning with LoRA/QLoRA, and LLM evaluation. It assumes you already know ML fundamentals and have built transformers from scratch.

---

# Section 8: Prompting Techniques

## Theory

### Zero-Shot Prompting
You describe the task in natural language and the model answers with no examples. This works because large language models are trained on vast corpora covering nearly every task format — they've seen classification, translation, summarisation, and reasoning described in prose millions of times. The model generalises from that pretraining without needing demonstrations. 

> Use zero-shot when the task is common, unambiguous, and the base model almost certainly has the pattern in its weights already (e.g. sentiment classification, simple Q&A, summarisation).

#### Zero-Shot — How It Actually Works

The model has a single set of weights — no task-specific heads. During pretraining it saw
sentiment analysis, translation, Q&A, and every other common task described in natural
language billions of times. Zero-shot works because the prompt tokens match those
pretraining patterns via attention, not because the model switches modes.

Every forward pass outputs a probability distribution over the full vocabulary (30k–100k tokens).
For binary classification, you're relying on the prompt context to push the highest-probability
token toward "POSITIVE" or "NEGATIVE".

In production you can make this more rigorous with **constrained decoding**: instead of
generating freely and parsing text, pull the raw logits for just your two target tokens,
softmax them against each other, and you get a clean binary probability. More reliable than
hoping the model doesn't output "This is POSITIVE" instead of just "POSITIVE".

### Few-Shot Prompting
You prepend 2–8 input/output examples before the real query. The model uses in-context learning — it pattern-matches the format and logic of your examples and applies them to the new input. Think of it like showing the model a few rows of a table and asking it to fill in the next row; it infers the column schema from what it sees. A handful of examples (3–5) is usually enough. More than 10 adds cost with diminishing returns unless your task is very idiosyncratic. 

> Use few-shot when zero-shot is inconsistent, when you need a very specific output format, or when the task has subtle rules that are easier to show than explain.

#### Few-Shot — In-Context Learning and the Architecture

When you add examples to the prompt, all of them — your examples and the final query —
are packed into a single forward pass as one long token sequence. There is no weight update,
no gradient, no fine-tuning. The model sees everything simultaneously as one long sequence. 

**Why examples help:** the attention mechanism lets every token attend to every other token
in the sequence. Your final token is attending to the example pairs before it, pulling
in the pattern: "when the input looks like X, the output looks like Y." The model is
inferring the task schema directly from the sequence structure.
    
    [ex1 input tokens] [ex1 label] [ex2 input tokens] [ex2 label] [your input tokens]

This is called **in-context learning** — the model's weights don't change, but the
activations do. The examples shift the internal representations at each layer, effectively
steering the model toward the demonstrated pattern without any training using attention.

**How many examples is enough:** 3–5 is usually sufficient. Beyond that you get diminishing
returns because the pattern is already clear from a few demonstrations. The main constraint
is context length — each example consumes tokens. More examples also means higher cost per
call.

**What few-shot actually buys you:** not new knowledge, but tighter format control and
disambiguation. If your task has subtle rules (like "mixed reviews are NEUTRAL, not
NEGATIVE"), it is far easier to demonstrate that with one example than to describe it
precisely in prose. The model picks up on implicit rules from demonstrations that would
be hard to articulate explicitly.

**Limitation:** in-context learning is ephemeral. The pattern exists only for that call.
The next call starts fresh. If you need the behaviour to persist reliably across all calls,
that is when fine-tuning becomes worth considering.

**What attention does**
At each layer, every token produces three vectors — Q, K, V — each of size d_k (e.g. 64
per head). The attention scores are computed as QK^T / sqrt(d_k), giving a (seq_len x seq_len)
matrix of weights. These weights are applied to the V vectors to produce an updated
representation for each token. Critically, the model uses **causal masking** — each token
can only attend to tokens before it in the sequence. So your final input tokens can attend
to all the example tokens that precede them, but not the other way around.

Each token ends up with its own representation (not one single vector) that has been
influenced by everything before it. The "in-context learning" signal lives in these
activations — the examples shift the internal representations at every layer, steering
the model toward the demonstrated pattern without any weight update.

**In-context learning**
No gradients, no weight updates, no kNN. The model's weights are frozen. The examples
work purely through attention — your input tokens attend back to the example pairs and
implicitly extract the pattern: "when input looks like X, output looks like Y." This
effect is ephemeral — it exists only for this call. The next call starts fresh.

### Chain-of-Thought (CoT)
Rather than asking for an answer directly, you ask the model to reason step by step before giving its final answer. Why does this help? Because autoregressive token generation means each token is conditioned on everything before it. When the model writes intermediate reasoning steps, those tokens become context for the final answer token — the model literally has more computation to draw on. It's analogous to showing your work in a maths exam: the act of writing it out forces coherent reasoning. CoT is most valuable on multi-step arithmetic, logical inference, and planning tasks. The phrase `"Let's think step by step"` is a reliable trigger.

### ReAct (Reason + Act)
ReAct interleaves reasoning traces with discrete actions (tool calls). The loop is: **Thought** → **Action** → **Observation** → **Thought** → ... until the model reaches a final answer. This matters for AI Engineers because it's the conceptual backbone of nearly every agent system. The model doesn't just generate text — it decides *what to do next*, does it (via a tool), sees the result, and updates its reasoning. No framework needed; you can implement ReAct with a plain `while` loop and an API call per iteration.

### Prompt Evaluation
Prompts are code — they need to be tested. Evaluation approaches range from:
- **Exact match / regex**: for structured outputs
- **Human eval**: gold standard but slow
- **LLM-as-a-Judge**: use a stronger model to score outputs (covered in Section 12)
- **Task-specific metrics**: BLEU for translation, F1 for extraction, etc.

The key discipline is: define your eval *before* you iterate on prompts, or you'll overfit to your intuition.

---

In [ ]:
# Install the Anthropic SDK
# %pip install anthropic --quiet

In [ ]:
import os
import anthropic

# Set your API key — either paste it here or set the ANTHROPIC_API_KEY env variable
# os.environ["ANTHROPIC_API_KEY"] = "your-key-here"

client = anthropic.Anthropic()  # Reads ANTHROPIC_API_KEY from environment
MODEL = "claude-haiku-4-5"  # Using claude-haiku-4-5 to keep costs low

def ask(prompt: str, max_tokens: int = 300) -> str:
    """Thin wrapper — sends a single user message and returns the text response."""
    # Using claude-haiku-4-5 to keep costs low
    response = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

print("Client ready.")

## 8.1 — Zero-Shot Prompting

In [ ]:
# Zero-shot: no examples, just task description
# Using claude-haiku-4-5 to keep costs low

zero_shot_prompt = """
Classify the sentiment of the following review as POSITIVE, NEGATIVE, or NEUTRAL.
Respond with only the label, nothing else.

Review: "The battery life is decent but the screen scratches way too easily."
"""

result = ask(zero_shot_prompt)
print(f"Zero-shot result: {result}")

Zero-shot result: NEGATIVE

The review said NEGATIVE. This is because there's no rules in place for mixed reviews. We can fix this using few shot next.

## 8.2 — Few-Shot Prompting
Same task, same review — but now we give the model examples of our labelling logic.

In [ ]:
# Few-shot: examples prime the model on our specific labelling conventions
# Using claude-haiku-4-5 to keep costs low

few_shot_prompt = """
Classify sentiment as POSITIVE, NEGATIVE, or NEUTRAL.
Mixed reviews that mention both pros and cons are NEUTRAL.
Respond with only the label.

Review: "Absolutely love this product, best purchase I've made."
Label: POSITIVE

Review: "Stopped working after two days. Complete waste of money."
Label: NEGATIVE

Review: "Camera is excellent but the software is buggy."
Label: NEUTRAL

Review: "The battery life is decent but the screen scratches way too easily."
Label:"""

result = ask(few_shot_prompt)
print(f"Few-shot result: {result}")
print()
print("Notice: with the explicit rule about mixed reviews, the model is more")
print("likely to return NEUTRAL than in the zero-shot case.")

Few-shot result: NEUTRAL

Same review but this time the model responded NEUTRAL. The third example demonstrated the mixed-review rule in a way the prose instruction alone didn't reliably enforce. Notice how its the same model used in zero shot and few shot. 

## 8.3 — Chain-of-Thought

In [ ]:
# Chain-of-thought: ask the model to reason before answering
# Using claude-haiku-4-5 to keep costs low

cot_prompt = """
A store sells apples for $0.50 each and oranges for $0.75 each.
Alice buys 4 apples and 3 oranges. She pays with a $5 bill.
How much change does she receive?

Think step by step before giving your final answer.
"""

result = ask(cot_prompt, max_tokens=400)
print(result)

Step 1: Calculate the cost of apples.
4 apples × $0.50 = $2.00

Step 2: Calculate the cost of oranges.
3 oranges × $0.75 = $2.25

Step 3: Calculate the total cost.
$2.00 + $2.25 = $4.25

Step 4: Calculate the change.
$5.00 - $4.25 = $0.75

Alice receives $0.75 in change.

## 8.4 — ReAct Loop (Manual, No Framework)

We'll implement a minimal ReAct loop from scratch. The model reasons, calls a mock tool, receives an observation, and reasons again. This is exactly what agent frameworks like LangChain do under the hood — we're just doing it explicitly.

In [ ]:
# Mock tool: a fake calculator the model can "call"
def mock_calculator(expression: str) -> str:
    """Safely evaluate a simple arithmetic expression."""
    try:
        # Only allow digits, spaces, and basic operators for safety
        allowed = set("0123456789 +-*/.()")
        if not all(c in allowed for c in expression):
            return "Error: invalid expression"
        result = eval(expression)  # Safe here — restricted to numbers/operators
        return str(result)
    except Exception as e:
        return f"Error: {e}"

print("Mock calculator ready.")
print("Test:", mock_calculator("4 * 0.50 + 3 * 0.75"))

In [ ]:
# ReAct loop — no framework, just a while loop and API calls
# Using claude-haiku-4-5 to keep costs low

REACT_SYSTEM = """
You are a reasoning agent. You have access to one tool:
  calculator(expression) — evaluates arithmetic and returns the result.

Respond in this exact format on each turn:
Thought: <your reasoning about what to do next>
Action: calculator(<arithmetic expression>)

When you have enough information to answer, respond with:
Thought: <final reasoning>
Answer: <your final answer>

Never skip the Thought line. Never call more than one action per turn.
"""

user_question = "A shirt costs $24.99. If there's a 15% discount and then 8% tax on the discounted price, what is the final price?"

messages = [{"role": "user", "content": user_question}]

print(f"Question: {user_question}")
print("=" * 60)

for step in range(6):  # Safety cap — real loops should also cap iterations
    # Using claude-haiku-4-5 to keep costs low
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=REACT_SYSTEM,
        messages=messages
    )
    reply = response.content[0].text
    print(f"\n[Step {step + 1}]\n{reply}")

    # Check if the model has reached a final answer
    if "Answer:" in reply:
        print("\n" + "=" * 60)
        print("ReAct loop complete.")
        break

    # Parse the action and execute the mock tool
    if "Action: calculator(" in reply:
        start = reply.index("calculator(") + len("calculator(")
        end = reply.index(")", start)
        expression = reply[start:end].strip()
        observation = mock_calculator(expression)
        print(f"Observation: {observation}")

        # Feed the observation back as an assistant + user turn
        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": f"Observation: {observation}"})
    else:
        # Model didn't call a tool — break to avoid an infinite loop
        print("No action found — ending loop.")
        break



============================================================

[Step 1]
Thought: I need to calculate the discounted price first by applying 15% off $24.99.
Action: calculator(24.99 * 0.85)
Observation: 21.2415

[Step 2]
Thought: Now I need to apply 8% tax to the discounted price of $21.2415.
Action: calculator(21.2415 * 1.08)
Observation: 22.9408

[Step 3]
Thought: I now have the final price after applying both the discount and tax.
Answer: The final price of the shirt after a 15% discount and 8% tax is $22.94.

============================================================
ReAct loop complete.

why is .content 0 here for index

## 8.5 — Your Turn: Experiment
Edit the cell below to try your own prompts. Try changing the task, the number of few-shot examples, or the CoT instruction.

In [ ]:
# ✏️ EXPERIMENT CELL — edit freely
# Using claude-haiku-4-5 to keep costs low

my_prompt = """
Your prompt here.
"""

print(ask(my_prompt))

## Notes

- The same model can handle zero shot, few shot, and react. These are just different ways of structuring the prompt. The model weights don't change at all. What changes is purely the input text. This is the whole point of in-context learning — one model, infinite task variations, just by rearranging the tokens it sees.

### Anthropic API Parameters — `client.messages.create`

#### Parameters used in Section 8

**`model`**
Which model to call. We use `claude-haiku-4-5` throughout — cheapest and fastest in the
Claude family, sufficient for all demos.

**`max_tokens`**
Hard cap on tokens the model can generate in its response. Generation stops the moment
this limit is hit, even mid-sentence. Does not control how many tokens are actually
produced — only the ceiling. Keep this low in demos to control costs.

**`messages`**
The conversation history as a list of `{"role": ..., "content": ...}` dicts.
Roles are `"user"` or `"assistant"`. The API is stateless — it remembers nothing between
calls. For multi-turn interactions (like the ReAct loop) you manually append each turn to
this list and resend the full history every call.

**`system`**
A special instruction that sits above the conversation and persists across every turn.
Sets the model's behaviour, persona, or rules without consuming a turn in `messages`.
Think of it as a zeroth message the model always sees but the user never does.

#### Common parameters not used in Section 8

**`temperature`**
Controls randomness in token sampling. `0` = greedy/deterministic (always picks the
highest probability token). `1` = default sampling. Higher = more random. For
classification tasks, set to `0` for consistent outputs.

**`stop_sequences`**
A list of strings that immediately halt generation when encountered. Useful in ReAct
loops — passing `["Observation:"]` stops the model after its action, preventing it
from hallucinating the observation itself.


## ✅ What You Just Learned — Section 8

**For AI Engineer interviews:**
- You can articulate *why* each technique works mechanistically, not just what it does
- You've implemented a ReAct loop from scratch — this demystifies LangChain/AutoGPT agents entirely
- Knowing when to use zero-shot vs. few-shot vs. CoT is a standard interview question; the answer is always "it depends on task complexity and consistency requirements"
- Prompt evaluation is often overlooked by candidates — mentioning it signals engineering maturity

---

# Section 9: Function Calling / Structured Outputs

Up until now, the model has just been generating text. You send a prompt, it sends back a string. That's it.
Function calling is the bridge between the model and the real world. The idea is: what if the model could decide to use a tool — like a calculator, a database, a weather API, a search engine — and tell your code what to call and with what arguments?
Here's the key insight that the theory section is trying to hammer home: the model never actually calls anything. It can't. It's a text generator sitting behind an API. What it can do is output a very specifically structured piece of JSON that says "I want to call this function with these arguments." Your Python code reads that JSON, calls the real function, gets the result, and sends it back. The model then uses that result to write its final answer.
So the full flow is:

- You define a tool in JSON (name, description, what arguments it takes)
- You send that schema to the model along with the user's question
- The model decides whether it needs the tool
- If yes, it returns a structured JSON blob saying which tool and what arguments
- Your code executes the real function
- You send the result back to the model
- The model writes the final response using that result

The reason this matters for AI Engineers is that this is how you connect LLMs to APIs, databases, search engines, code interpreters — anything. Every "AI agent" you've ever heard of is built on this loop.

Note: An API (Application Programming Interface) is just a way for two pieces of software to talk to each other.
The easiest way to think about it: a restaurant analogy. You're the customer, the kitchen is some service with data or functionality you want, and the waiter is the API. You don't go into the kitchen yourself — you tell the waiter what you want, the waiter goes to the kitchen, and comes back with the result.
Some concrete examples:

When your phone shows the weather, it's calling a weather API — sending a request with your location, getting back temperature and forecast data
When you pay with Stripe on a website, the website is calling Stripe's API — sending card details, getting back "payment successful"
When we call client.messages.create(...) in our notebook, we're calling Anthropic's API — sending a prompt, getting back a response

In all cases the pattern is the same: you send a request with some inputs, you get back a response with some outputs. You don't know or care what's happening inside — you just use the result.
In the context of Section 9, when we build a "weather tool" or "calculator tool", we're simulating what it would look like if the model could trigger a call to a real external API. In production that might be hitting the OpenWeatherMap API to get real weather data, or querying your company's database. In our demo we just fake the result so we don't need any external accounts or keys.

## Theory

### What Actually Happens Under the Hood
The model never executes code. When you define a tool, you're giving the model a JSON schema describing what arguments a function accepts. The model reads the schema, decides whether to call the tool, and if so, returns a structured JSON object specifying which tool to call and with what arguments. Your Python code then parses that JSON, calls the real function, and sends the result back in the next turn. The model is a text generator that happens to output valid JSON when given the right schema context.

### Tool Schemas
A tool schema is just a JSON Schema object describing: `name`, `description`, and `input_schema` (the parameters). The description is critical — it's what the model reads to decide whether to use the tool. A vague description leads to misuse or non-use.

### Tool Selection
The model reads the user's message alongside all tool descriptions and decides: should I call a tool, and if so, which one? This is a classification decision baked into the model's RLHF training. You can influence it by writing clear descriptions and, when needed, adding explicit instructions in the system prompt like `"Always use the search tool before answering factual questions."`

### The Execution Loop
1. Send messages + tool schemas to the API
2. Model returns a `tool_use` content block with `name` and `input`
3. Your code executes the real function with those inputs
4. Send the result back as a `tool_result` block
5. Model generates its final user-facing response using the result

### Why the Model Never "Calls" Anything
This is a common misconception worth correcting in interviews. The model outputs structured text. The developer's code is what calls functions, hits APIs, or queries databases. The model is stateless — it just sees tokens.

### Structured Outputs vs. Tool Use

Both involve JSON passing between the model and your code — which is why they're easy
to confuse. The difference is what that JSON *means* and what you do with it.

**Tool use** is the model deciding to take an action mid-conversation. It outputs a JSON
blob saying "call this function with these arguments." Your code executes something real
(hits an API, queries a database, runs a calculation) and sends the result back. There is
an execution loop. The model is driving a process.

**Structured outputs** has no execution loop. You're asking the model to format its
response as JSON instead of prose. The model reads something, thinks about it, and writes
its answer in a structured format your code can parse. No tool, no external call, nothing
sent back.

| | Tool Use | Structured Outputs |
|---|---|---|
| **Purpose** | Trigger an action | Format a response |
| **Execution loop** | Yes | No |
| **External call** | Yes | No |
| **Use when** | You need fresh or external data | You need parseable output from what the model already knows |

A useful way to remember it: tool use is the model saying "I need to go get something."
Structured output is the model saying "here is my answer, formatted so your code can
read it."

In production you often use both together — a tool call fetches raw data from an API,
then you ask the model to return its analysis as structured JSON so your frontend can
render it cleanly.

Knowing when to use each is a common AI Engineer interview question. Reaching for tool
use when structured outputs would do adds unnecessary complexity, an extra API call, and
more code to maintain.

### Validation
## Validation

Validation is checking that the data you received is what you actually expected before
you act on it.

This matters specifically in function calling because the model is generating the function
arguments — and the model can hallucinate. It might produce an argument in the wrong
format, an impossible value, or something that would break your code or cause a real-world
problem if you acted on it blindly.

Common failure modes:
- **Wrong type**: model returns `"twenty dollars"` where you expected a number
- **Impossible value**: model returns a negative price on a payment tool
- **Wrong format**: model returns `"March 15th 2024"` where your database expects `YYYY-MM-DD`
- **Missing field**: model omits a required argument entirely

The mental model: treat model-generated JSON exactly like user input from a web form.
You wouldn't pass a number a user typed directly into a database query without checking
it. Same principle applies here.

In practice this means checking that required fields exist, that values are the right
type, and that values are within acceptable ranges before executing anything:

```python
def call_calculator(args: dict) -> str:
    if "expression" not in args:
        return "Error: missing expression"
    if not isinstance(args["expression"], str):
        return "Error: expression must be a string"
    allowed = set("0123456789 +-*/.() ")
    if not all(c in allowed for c in args["expression"]):
        return "Error: invalid characters in expression"
    return str(eval(args["expression"]))
```

In production you would typically use **Pydantic** to define a schema and validate against
it in one step rather than writing manual checks. Pydantic lets you declare the expected
shape of your data as a Python class, and raises a clear error the moment the model's
output doesn't conform to it. The principle is the same — never trust model-generated
arguments until they've been checked.

The short version: the model is a probabilistic text generator that outputs valid JSON
most of the time. Most of the time is not good enough when you're hitting a real API or
writing to a database.

---

## 9.1 — Define a Tool

In [ ]:
# Define the tool schema (what we send to the model)
# and the actual Python function (what we run when the model calls it)

calculator_tool_schema = {
    "name": "calculator",
    "description": (
        "Evaluates a mathematical expression and returns the numeric result. "
        "Use this whenever you need to perform arithmetic."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "A valid arithmetic expression, e.g. '2 * (3 + 4)'"
            }
        },
        "required": ["expression"]
    }
}

def calculator(expression: str) -> str:
    """The real function — validate and execute."""
    allowed = set("0123456789 +-*/.()")
    if not all(c in allowed for c in expression):
        return "Error: expression contains invalid characters"
    try:
        return str(round(eval(expression), 4))
    except Exception as e:
        return f"Error: {e}"

print("Tool schema and function defined.")

What's happening:

We're defining two completely separate things that happen to be related:
The schema is a JSON object we'll send to the model. It tells the model three things — what the tool is called, what it does (the description is critical, this is what the model reads to decide whether to use it), and what arguments it accepts. Notice input_schema is just a standard JSON Schema object describing the parameters. The "required": ["expression"] line tells the model this argument is not optional.

The function is real Python code that only we run — the model never sees it. It takes an expression string, validates the characters first (our validation step), then uses Python's built-in eval() to compute the result. We restrict characters to only numbers and basic operators so eval() can't execute anything dangerous.

These two things are intentionally separate. The schema is the model's view of the tool. The function is the real implementation. They need to match — same argument names, same types — but they live in different worlds.

## 9.2 — Full Function-Calling Loop

In [ ]:
# Full tool-use loop: schema → model returns tool_use → execute → send result → final response
# Using claude-haiku-4-5 to keep costs low

def run_tool_loop(user_message: str) -> str:
    messages = [{"role": "user", "content": user_message}]

    while True:
        # Using claude-haiku-4-5 to keep costs low
        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            tools=[calculator_tool_schema],
            messages=messages
        )

        # Append the assistant's response to the conversation
        messages.append({"role": "assistant", "content": response.content})

        # If the model is done (no more tool calls), return the text
        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text

        # Otherwise, process all tool_use blocks
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  → Model called: {block.name}({block.input})")
                result = calculator(**block.input)  # Execute the real function
                print(f"  ← Result: {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })

        # Send results back to the model
        messages.append({"role": "user", "content": tool_results})


print("--- Tool loop demo ---")
question = "What is 15% of 847, and then subtract 23.5 from that?"
print(f"User: {question}")
answer = run_tool_loop(question)
print(f"\nFinal answer: {answer}")

#### How the loop works

Every API response has exactly two attributes we care about:

- `response.stop_reason` — why the model stopped generating. Drives the logic of the
  loop. Four possible values: `"end_turn"` (done), `"tool_use"` (wants a tool),
  `"max_tokens"` (got cut off), `"stop_sequence"` (hit a stop string).
- `response.content` — a list of content blocks containing what the model produced.

The model never returns a plain string. It returns a list of **blocks**, each with a
`type` field. Two types matter here:

- `TextBlock` — has a `.text` attribute containing the model's response as a string
- `ToolUseBlock` — has `.name` (tool to call), `.input` (arguments as a dict),
  and `.id` (unique identifier for this specific call)

The reason it's a list is that the model can return multiple blocks in one response —
for example a text block followed by a tool_use block, or multiple tool_use blocks if
it decides to call several tools at once.

The loop control flow, stripped to its skeleton:

```python
while True:
    response = client.messages.create(...)

    if response.stop_reason == "end_turn":
        return block.text      # ← only exit point, kills the loop

    # if we're still here, stop_reason was "tool_use"
    # execute the tool, append result to messages, loop again
```

`return block.text` is the only way out of the loop. In Python, hitting a `return`
inside a `while True` inside a function exits the function entirely. Every iteration
either exits via `return` or falls through to tool execution and loops back to the top.
In production always add a safety cap (e.g. `for step in range(6)`) so a misbehaving
model can't loop forever.

#### What the raw response looks like on each iteration

**Iteration 1 — model wants to call the calculator:**
```python
response.stop_reason = "tool_use"
response.content = [
    ToolUseBlock(
        type="tool_use",
        id="toolu_01ABC123",
        name="calculator",
        input={"expression": "847 * 0.15"}
    )
]
```
We extract `block.input`, run `calculator("847 * 0.15")`, get back `"127.05"`.
Messages list after appending:
```python
messages = [
    {"role": "user",      "content": "What is 15% of 847, and then subtract 23.5?"},
    {"role": "assistant", "content": [ToolUseBlock(id="toolu_01ABC123", ...)]},
    {"role": "user",      "content": [{"type": "tool_result",
                                       "tool_use_id": "toolu_01ABC123",
                                       "content": "127.05"}]}
]
```

**Iteration 2 — model wants to call the calculator again:**
```python
response.stop_reason = "tool_use"
response.content = [
    ToolUseBlock(
        type="tool_use",
        id="toolu_01DEF456",
        name="calculator",
        input={"expression": "127.05 - 23.5"}
    )
]
```
We run `calculator("127.05 - 23.5")`, get back `"103.55"`.
Messages list after appending:
```python
messages = [
    {"role": "user",      "content": "What is 15% of 847, and then subtract 23.5?"},
    {"role": "assistant", "content": [ToolUseBlock(id="toolu_01ABC123", ...)]},
    {"role": "user",      "content": [{"type": "tool_result",
                                       "tool_use_id": "toolu_01ABC123",
                                       "content": "127.05"}]},
    {"role": "assistant", "content": [ToolUseBlock(id="toolu_01DEF456", ...)]},
    {"role": "user",      "content": [{"type": "tool_result",
                                       "tool_use_id": "toolu_01DEF456",
                                       "content": "103.55"}]}
]
```

**Iteration 3 — model is done:**
```python
response.stop_reason = "end_turn"
response.content = [
    TextBlock(
        type="text",
        text="15% of 847 is 127.05, and subtracting 23.5 from that gives you 103.55."
    )
]
```
`stop_reason == "end_turn"` → we find the `TextBlock` → `return block.text` → loop exits.

The `tool_use_id` is what ties results to calls. With multiple tool calls in the history
the model needs to know which result belongs to which call. The ID is how it knows.
Messages grow on every iteration — by the final call the model sees the entire chain:
original question, all tool calls, all results.

## 9.3 — Structured Output Extraction (No Tool Use)

In [ ]:
# Structured output: force JSON format via prompting (no tool invocation)
# Using claude-haiku-4-5 to keep costs low

import json

text_to_extract = """
Order #8821: Jane Smith ordered 3x Wireless Headphones at $89.99 each and
1x USB-C Cable at $12.50. Shipping to Toronto, ON. Order placed 2024-03-15.
"""

extraction_prompt = f"""
Extract the order details from the text below and return ONLY a JSON object.
No markdown, no explanation, just the raw JSON.

Required fields:
- order_id (string)
- customer_name (string)
- items (list of objects with: name, quantity, unit_price)
- shipping_city (string)
- order_date (string, YYYY-MM-DD)
- total (float, calculated)

Text:
{text_to_extract}
"""

# Using claude-haiku-4-5 to keep costs low
raw = ask(extraction_prompt, max_tokens=400)

try:
    parsed = json.loads(raw)
    print("Parsed successfully:")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Parse error: {e}")
    print(f"Raw response: {raw}")

No tool schema, no execution loop, no result sent back. This is structured output —
we are asking the model to write its answer as JSON instead of prose.

The prompt does two things: tells the model exactly what fields to include and what
types they should be, and explicitly says "return ONLY a JSON object, no markdown, no
explanation." That second part is important — without it the model often wraps the JSON
in markdown code fences which breaks `json.loads()`.

After getting the response we immediately parse it with `json.loads()`, which converts
the raw JSON string into a Python dict. The `try/except` is our validation step — if
the model returned something that isn't valid JSON we catch the error gracefully instead
of crashing.

Notice we asked the model to calculate the total itself. It had all the information it
needed in the prompt so no tool was required. This is the structured output vs tool use
distinction in practice — no external data needed means no tool needed.

## 9.4 — Your Turn: Define Your Own Tool

In [ ]:
# ✏️ EXPERIMENT CELL — define your own tool and test it
# Using claude-haiku-4-5 to keep costs low

# Example starter: a unit converter tool
my_tool_schema = {
    "name": "convert_units",
    "description": "Converts a value from one unit to another. Supports km/miles, kg/lbs, C/F.",
    "input_schema": {
        "type": "object",
        "properties": {
            "value": {"type": "number", "description": "The numeric value to convert"},
            "from_unit": {"type": "string", "description": "Source unit (km, miles, kg, lbs, C, F)"},
            "to_unit": {"type": "string", "description": "Target unit"}
        },
        "required": ["value", "from_unit", "to_unit"]
    }
}

def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    conversions = {
        ("km", "miles"): lambda v: v * 0.621371,
        ("miles", "km"): lambda v: v * 1.60934,
        ("kg", "lbs"): lambda v: v * 2.20462,
        ("lbs", "kg"): lambda v: v * 0.453592,
        ("C", "F"): lambda v: v * 9/5 + 32,
        ("F", "C"): lambda v: (v - 32) * 5/9,
    }
    fn = conversions.get((from_unit, to_unit))
    if fn:
        return f"{round(fn(value), 4)} {to_unit}"
    return f"Unsupported conversion: {from_unit} → {to_unit}"


# Test it manually
print(convert_units(100, "km", "miles"))

# Then wire it up to the model — replace the schema and function name in run_tool_loop
# or write your own loop below

## ✅ What You Just Learned — Section 9

**For AI Engineer interviews:**
- You can explain that the model outputs JSON — it doesn't execute anything. This is a common misconception and correcting it signals depth
- You've built the tool-use loop from scratch — no magic, just message passing
- Structured outputs vs. tool use is a nuanced distinction that frequently comes up: one is about constrained generation, the other is about agentic action
- Validation of model-generated JSON is a production concern most candidates miss

---

# Section 10: Fine-Tuning vs. RAG vs. Prompting — When to Use Each

## Decision Framework

This question comes up constantly in AI Engineer interviews because it's the first architectural decision you make on any LLM project. The wrong choice wastes weeks and significant money.

### Prompt Engineering
**Use when:** the base model already has the required knowledge and just needs guidance on format, tone, or task framing. This is always your first attempt. It's free to iterate, requires no infrastructure, and can be deployed in minutes. Limitations: fragile under distribution shift, can't inject proprietary knowledge reliably, context window limits apply.

### Retrieval-Augmented Generation (RAG)
**Use when:** the model lacks knowledge — either because the information is proprietary (internal docs, codebases, customer data) or because it post-dates the training cutoff. RAG retrieves relevant chunks at inference time and puts them in the prompt. No retraining, no model weights change. Limitations: retrieval quality is a hard dependency; garbage in, garbage out. Latency increases. Context windows can fill up.

### Fine-Tuning
**Use when:** you need the model to reliably produce a very specific output style, format, or behaviour that you cannot achieve through prompting alone — and you have labelled examples. Fine-tuning bakes patterns into the weights. It doesn't add new factual knowledge (the model can still hallucinate facts it never saw in training) but it does instil consistent behaviour. Requires data curation, compute, and evaluation infrastructure. Most teams reach for fine-tuning too early.

## Decision Table

| Dimension | Prompt Engineering | RAG | Fine-Tuning |
|---|---|---|---|
| **Cost** | Lowest (just tokens) | Medium (retrieval infra + tokens) | Highest (training compute + storage) |
| **Latency** | Baseline | Higher (retrieval adds a step) | Baseline (no retrieval needed) |
| **Data Required** | None | Documents / knowledge base | Labelled input-output pairs |
| **Iteration Speed** | Minutes | Hours (index rebuild) | Days to weeks |
| **Adds New Knowledge** | No (relies on pretraining) | Yes (retrieved at runtime) | Marginal (patterns, not facts) |
| **Use When** | Base model has the knowledge | Knowledge is missing or private | Style/format/behaviour consistency needed |
| **Risk** | Prompt injection, fragility | Retrieval failures, context length | Overfitting, catastrophic forgetting |

## Why This Comes Up in Interviews
AI Engineer roles sit at the intersection of ML and product. Interviewers want to know you won't waste three months fine-tuning a model when a two-line system prompt would have solved it. Demonstrating this judgment — and the ability to articulate the tradeoffs — is what separates engineers who ship from engineers who research indefinitely.

The answer to "which should I use?" is almost always: **prompt first, RAG if you need fresh/private knowledge, fine-tune only if prompting + RAG can't get you to the required reliability bar**.

---

# Section 11: PEFT and LoRA / QLoRA Fine-Tuning

Note - in section 8/9 we were using APIs. here we're actually loading a model and fine tuning it. 

## Theory

### Why Full Fine-Tuning is Memory-Prohibitive
Full fine-tuning updates every parameter in the model. For a 7B parameter model, storing the weights in fp32 requires ~28 GB of GPU memory — before gradients (~28 GB more) and optimizer states (Adam keeps two moment estimates: another ~56 GB). That's 112 GB total for a 7B model. Most practitioners don't have that. PEFT (Parameter-Efficient Fine-Tuning) methods solve this by only training a tiny fraction of parameters.

### Trainable Parameters
PEFT freezes the base model's weights entirely. Only the adapter parameters — a tiny fraction of the total — are updated during training. For LoRA with r=8 on a medium model, this is often less than 1% of total parameters. Gradients and optimizer states are only needed for these small matrices, which slashes memory requirements dramatically.

### LoRA: Low-Rank Adaptation
Instead of modifying the weight matrix `W` directly, LoRA inserts two small matrices: `W' = W + α * B @ A`, where `A` has shape `(r, d_in)` and `B` has shape `(d_out, r)`. Only `A` and `B` are trained; `W` stays frozen. The rank `r` controls the capacity of the adapter — higher rank = more expressiveness but more memory. `r=8` is a solid starting point for most tasks; try `r=16` if you see underfitting.

#### LoRA — what's actually happening mathematically

When you normally fine-tune a model, you update a weight matrix W directly. For a layer in a transformer, W might be shape (768, 768) — that's 589,824 numbers to update, store gradients for, and keep optimizer states for.
LoRA says: instead of updating W, freeze it completely and add a side path:

output = W @ input + (B @ A) @ input

Where A is shape (r, 768) and B is shape (768, r). With r=8, that's just 8×768 + 768×8 = 12,288 numbers — about 48x fewer than updating W directly. Only A and B have gradients. W is frozen and never changes.
The product B @ A is always a (768, 768) matrix — same shape as W — so the math works out identically, you're just constraining the update to live in a low-rank subspace. The intuition is that the meaningful changes needed for fine-tuning are actually low-rank — you don't need the full expressiveness of a 768×768 update.

At inference time you can either keep them separate (load base + adapter) or merge them: W_merged = W + B @ A, giving you a single matrix with zero overhead.

Rank controls the capacity of the adapter. r=8 means the update matrix can only represent 8 linearly independent directions of change. That's enough for most tasks. r=16 doubles it. Going higher starts to approach full fine-tuning in cost and risk of overfitting.

alpha (lora_alpha) is a scaling factor — the actual update applied is (alpha/r) * B @ A. Setting alpha=2r (so alpha=16 when r=8) is a common convention that keeps the effective learning rate stable as you change rank.

### QLoRA: Quantised LoRA
QLoRA adds one thing on top: the frozen base weights (W) are stored in 4-bit quantisation rather than 32-bit or 16-bit floats. A 768×768 matrix in fp32 takes 2.4 MB. In 4-bit it takes 0.3 MB — 8x smaller.
The adapters A and B stay in higher precision (bf16 or fp32) because they're being trained and need precision. But since the base weights are frozen, you can aggressively compress them.

This is what makes fine-tuning 7B+ models possible on a single consumer GPU. In our notebook we're using opt-125m on CPU so QLoRA isn't necessary — but the concept is the same, just without the load_in_4bit=True flag.

### What Fine-Tuning Produces
The output is **adapter weights** — a small file (often a few MB for small models) that stores the `A` and `B` matrices. The base model weights are unchanged. To use the fine-tuned model, you load the base model and merge or apply the adapter on top. This means you can share adapters without redistributing the full model.

### Choosing Rank
- `r=4`: very lightweight, for simple style or format tasks
- `r=8`: good default for most tasks
- `r=16`: more capacity, use if r=8 underfits
- `r=32+`: rarely needed; at this point full fine-tuning may be more efficient

---

In [1]:
#Just installing the four libraries. 
# transformers is HuggingFace's model library. 
# peft is the LoRA/adapter library, also from HuggingFace. datasets lets you load standard datasets in one line. 
# bitsandbytes handles 4-bit quantisation for QLoRA — not needed for CPU but installed so the import doesn't fail.

#  Install required packages — quiet flag keeps notebook output clean
# !pip install transformers peft datasets --quiet
# bitsandbytes is for QLoRA quantisation — may be skipped if no GPU
# !pip install bitsandbytes --quiet

## Code

NOTE about Decoder only translation model

OPT-125m is a decoder-only model, not a sequence-to-sequence model.

Translation is classically a seq2seq task. The original approach used an encoder-decoder
architecture — the encoder reads the source sentence and builds a representation, the
decoder generates the target sentence conditioned on that representation. Models like
T5 and BART work this way.

Decoder-only models like OPT, GPT, and Claude can also do translation but differently.
Instead of encoding source and decoding target as two separate processes, they treat the
whole thing as next token prediction. The task is framed as a prompt:

Translate from english to french: 'Good morning'

The model just continues generating tokens, which happen to be French. It learned during
pretraining that this prompt pattern is followed by a translation, so it reproduces that
pattern.

| | Encoder-Decoder (T5, BART) | Decoder-Only (OPT, GPT) |
|---|---|---|
| **Architecture** | Two separate stacks | Single stack |
| **Translation** | Built for it | Treats it as text completion |
| **General tasks** | Less flexible | Handles anything as a prompt |
| **Efficiency** | Better for pure translation | Better for general purpose use |

The modern LLM world has largely converged on decoder-only because scaling laws turned
out to favour them for general purpose tasks. You don't need a specialised architecture
for translation anymore when a large enough decoder-only model can do it from a prompt.

OPT-125m can do translation — just not very well at this size. A proper seq2seq model
like T5 fine-tuned on the same data would produce better translations. But for
demonstrating LoRA the architecture choice doesn't matter — we just need something small
and safe to train locally.

In [1]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device = "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.2.2


In [3]:
import sys
print(sys.executable)

/Users/Sarah/Documents/Programming/MMAI Python Bootcamp/bootcamp_env_py312/bin/python


## 11.1 — Load a Small Base Model
I'll use `facebook/opt-125m` — 125M parameters, safe for my laptop.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "facebook/opt-125m"  # 125M params — the maximum we'll load locally

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32  # fp32 for CPU safety
)
base_model = base_model.to(device)

total_params = sum(p.numel() for p in base_model.parameters())
print(f"Total parameters: {total_params:,} ({total_params / 1e6:.1f}M)")

In [ ]:
for p in base_model.parameters():
    print(p)

What's happening:

Two things get loaded here — the tokenizer and the model itself.

The tokenizer converts raw text into token IDs and back. It's a separate object from the model because the same tokenizer can be used across different model sizes in the same family. AutoTokenizer.from_pretrained figures out which tokenizer class to use based on the model name and downloads it from HuggingFace.

The model is loaded with AutoModelForCausalLM — the ForCausalLM part means it's set up for causal language modelling, i.e. predicting the next token. from_pretrained downloads the weights from HuggingFace and caches them in ~/.cache/huggingface/hub/. It only downloads once — subsequent runs use the cache.

torch_dtype=torch.float32 tells it to load weights in 32-bit precision. We use this instead of float16 because CPU doesn't handle float16 well. On a CUDA GPU you'd use float16 or bfloat16 to halve memory usage.

.to(device) moves all the model weights onto whichever device I set.

The parameter count line iterates over every parameter tensor in the model, calls .numel() to get the number of elements in that tensor, and sums them all up. 125 million is the ceiling we set in our safety rules — never go above this locally.

## 11.2 — Baseline Generation (Before Fine-Tuning)

tokenizer(prompt, return_tensors="pt") converts your text string into a tensor of token IDs — the numbers the model actually sees. 

torch.no_grad() tells PyTorch not to build a computation graph, saving memory since we're just doing inference. 

model.generate() runs the forward pass repeatedly, picking one token at a time. We then slice off just the new tokens (not the prompt) and decode them back to text. This gives you a baseline to compare against after fine-tuning.

In [ ]:
def generate(model, prompt: str, max_new_tokens: int = 60) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Greedy decoding for reproducibility
            pad_token_id=tokenizer.eos_token_id
        )
    # Decode only the newly generated tokens
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


test_prompt = "Translate English to French: 'The weather is beautiful today.'"
print("Prompt:", test_prompt)
print("Before fine-tuning:", generate(base_model, test_prompt))

Prompt: Translate English to French: 'The weather is beautiful today.'
Before fine-tuning: 

Thursday, October 30, 2010

The weather is beautiful today.

I have been thinking about this for a while now. I have been thinking about it for a while now. I have been thinking about it for a while now. I have been thinking about it for a while


What's happening:

We define a generate function we'll reuse after fine-tuning to compare outputs.

tokenizer(prompt, return_tensors="pt") converts the prompt string into a dict containing a tensor of token IDs. The "pt" means return PyTorch tensors rather than plain lists. .to(device) moves those tensors to the right device.

torch.no_grad() is a context manager that tells PyTorch not to track gradients during this forward pass. We're just generating, not training, so we don't need gradients. This saves memory and speeds things up.

do_sample=False means greedy decoding — always pick the highest probability token at each step. This makes generation deterministic and reproducible, which is what we want for a before/after comparison.

The slicing line output[0][inputs["input_ids"].shape[1]:] is how we extract only the newly generated tokens. The model returns the full sequence including the prompt tokens, so we skip past the prompt length to get just the new content.

tokenizer.decode converts the token IDs back into a human readable string. skip_special_tokens=True removes any padding or end-of-sequence tokens from the output.

Notice the output isn't great French — OPT-125m is a tiny general purpose model, not a translation model. That's expected. The point is to see whether it improves after fine-tuning.

## 11.3 — Apply LoRA Config

target_modules=["q_proj", "v_proj"] tells LoRA which weight matrices to inject adapters into — specifically the query and value projection matrices inside each attention layer. These are the most impactful for style/behaviour changes. It's not just the last layer — it injects adapter matrices into every attention layer in the model. So if opt-125m has 12 transformer layers, you get 12 pairs of LoRA adapters, one per layer's Q and V projections.
Why Q and V and not K? Convention based on the original LoRA paper. The intuition is that Q (what am I looking for) and V (what do I retrieve) have the most influence over what the attention mechanism actually learns to attend to and return. K (what do I offer) matters less for adaptation. That said, you can add K too — some configs do ["q_proj", "k_proj", "v_proj"].
Also worth noting — opt-125m is a decoder-only model, same family as GPT. There's no encoder. So all 12 layers are decoder self-attention layers, and LoRA patches Q and V in all 12. 

get_peft_model wraps your base model, freezes all the original weights, and inserts the A and B matrices at the specified locations. After this call, only the adapter parameters have requires_grad=True.

lora_dropout=0.05 — drops 5% of adapter activations to zero randomly during training, same concept as regular dropout. Mild regularisation to reduce overfitting. With only 20 steps and 80 examples it barely matters, but it's good practice.

bias="none" — tells LoRA not to train any bias terms, only the A and B matrices. Options are "none", "all", or "lora_only". "none" is the standard starting point and keeps the parameter count as low as possible.

task_type=TaskType.CAUSAL_LM — tells peft what kind of model this is so it knows how to wire up the adapter correctly. Other options include SEQ_2_SEQ_LM for encoder-decoder models like T5, SEQ_CLS for classifiers. This affects which modules peft targets internally.

In [6]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,                          # Rank — controls adapter capacity
    lora_alpha=16,                # Scaling factor (common to set to 2*r)
    target_modules=["q_proj", "v_proj"],  # Which weight matrices to adapt
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

lora_model = get_peft_model(base_model, lora_config)

# Show how few parameters we're actually training
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in lora_model.parameters())
print(f"Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}% of total)")
print(f"Frozen parameters:    {total - trainable:,}")

'NoneType' object has no attribute 'cadam32bit_grad_fp32'
Trainable parameters: 294,912 (0.23% of total)
Frozen parameters:    125,239,296


/Users/Sarah/Documents/Programming/MMAI Python Bootcamp/bootcamp_env_py312/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


Trainable parameters: 294,912 (0.24% of total)
Frozen parameters:    124,944,384

What's happening:

This is the core of LoRA. get_peft_model takes our base model and wraps it — it freezes all the original weights and injects small trainable adapter matrices into the layers we specified.
The parameters in LoraConfig:

r=8 — the rank of the adapter matrices. Every targeted weight matrix W gets two small matrices injected: A of shape (r, d_in) and B of shape (d_out, r). The update is W + alpha/r * B @ A. Lower rank = fewer parameters = less capacity. r=8 is a solid default
lora_alpha=16 — the scaling factor. Setting it to 2*r is a common convention. Controls how much the adapter influences the output relative to the frozen weights
target_modules=["q_proj", "v_proj"] — which weight matrices to inject adapters into. We're targeting the query and value projection matrices inside the attention layers. These are the most impactful for behaviour change. You could also add "k_proj" and "out_proj" for more capacity
lora_dropout=0.05 — small dropout on the adapter for regularisation, helps prevent overfitting on tiny datasets
bias="none" — don't train bias terms, keeps the adapter minimal
task_type=TaskType.CAUSAL_LM — tells PEFT this is a causal language model so it sets things up correctly

The parameter count at the end shows the dramatic difference — we went from 125M total parameters down to only 294,912 trainable ones. That's 0.24% of the model. Gradients and optimizer states only need to be computed for those 294k parameters, which is why LoRA fits in memory where full fine-tuning wouldn't.

## 11.4 — Load a Tiny Dataset

load_dataset pulls a standard translation dataset from HuggingFace. split="train[:80]" takes only the first 80 examples — enough to show the fine-tuning loop without running forever. Each example is a dict with an "en" and "fr" string.

In [ ]:
from datasets import load_dataset

# Use a tiny slice of a translation dataset — 80 training examples maximum
dataset = load_dataset("Helsinki-NLP/opus-100", "en-fr", split="train[:80]")

print(f"Dataset size: {len(dataset)} examples")
print("Sample:", dataset[0])

Dataset size: 80 examples
Sample: {'translation': {'en': 'Hello, how are you?', 'fr': 'Bonjour, comment allez-vous?'}}

What's happening:

load_dataset downloads a dataset from HuggingFace. We're using Helsinki-NLP/opus-100, a large multilingual translation dataset. The "en-fr" argument specifies the English-French language pair.

split="train[:80]" is the important part — we're taking only the first 80 examples from the training split. The full dataset has millions of examples. We only need enough to demonstrate that fine-tuning changes the model's behaviour, not enough to actually make it a good translator.

Each example is a dict with a translation key containing both the English and French versions. We'll format these into training prompts in the next cell.

In [ ]:
# Tokenise the dataset
def tokenize(example):
    en = example["translation"]["en"]
    fr = example["translation"]["fr"]
    text = f"Translate English to French: '{en}' => '{fr}'"
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(tokenize, remove_columns=dataset.column_names)
tokenized.set_format("torch")
print(f"Tokenised {len(tokenized)} examples.")

Map: 100%|██████████| 80/80 [00:00<00:00, 814.22 examples/s]

Tokenised 80 examples.


What's happening:

We convert each raw text example into tensors the model can train on.

The tokenize function formats each example as a prompt-completion pair in the same style as our test prompt — "Translate English to French: '{en}' => '{fr}'". The model learns to associate this input format with French output.

truncation=True cuts sequences longer than max_length=128 tokens. padding="max_length" pads shorter sequences to exactly 128 tokens so all examples in a batch are the same length — this is required for batching.

tokens["labels"] = tokens["input_ids"].copy() is a causal LM training convention. The labels are the same as the input IDs — we're training the model to predict each token given all previous tokens. The loss function internally shifts these by one position so the model is predicting the next token at each step.

dataset.map applies the tokenize function to every example. remove_columns drops the original text columns since we no longer need them. set_format("torch") converts everything to PyTorch tensors.

## 11.5 — Fine-Tuning Loop

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Safety: fp16 only if GPU is available
use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="/tmp/lora_adapter",   # Temporary — won't clutter your filesystem
    num_train_epochs=2,               # Hard cap: never more than 2 epochs in demos
    per_device_train_batch_size=1,    # Memory safety
    gradient_accumulation_steps=2,    # Effective batch size = 2
    max_steps=20,                     # Hard cap: 20 steps maximum
    learning_rate=3e-4,
    fp16=use_fp16,
    logging_steps=5,
    save_strategy="no",               # Don't auto-save checkpoints to disk
    report_to="none",                 # No wandb or mlflow
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

print("Starting fine-tuning (20 steps max)...")
trainer.train()
print("Fine-tuning complete.")

What's happening:

`TrainingArguments` is a configuration object. Let's go through each parameter:

`output_dir` ="/tmp/lora_adapter" — where to write checkpoints. We use /tmp/ so nothing persists permanently on your machine
`num_train_epochs` = 2 — hard cap at 2 epochs as per our safety rules. One epoch means the model has seen every training example once
`per_device_train_batch_size` = 1 — process one example at a time. Memory safety rule — keeps RAM usage minimal
`gradient_accumulation_steps` = 2 — instead of updating weights after every example, accumulate gradients over 2 steps before updating. Effective batch size becomes 2 without needing more memory
`max_steps`=20 — hard cap at 20 weight updates regardless of epochs. This overrides the epoch setting if epochs would require more steps
`learning_rate`=3e-4 — how large each weight update step is. Standard starting point for LoRA
`fp16`=use_fp16 — half precision training, only enabled if CUDA is available. False on your Mac
`logging_steps`=5 — print a loss update every 5 steps so you can see training progressing
`save_strategy`="no" — don't auto-save checkpoints during training, only save when we explicitly call it
`report_to`="none" — don't send metrics to Weights & Biases or any other experiment tracker

`DataCollatorForLanguageModeling` handles batching — it takes individual examples and combines them into batches. 
`mlm`=False means causal language modelling, not masked language modelling like BERT.

`Trainer` is HuggingFace's high level training loop. It handles the forward pass, loss calculation, backward pass, and weight update for you. Under the hood it's doing exactly what you'd write manually in PyTorch — it's just a lot less code.

The loss numbers in the output should decrease across steps, showing the adapter is learning. Don't expect it to get very low — 20 steps on 80 examples is minimal training, just enough to demonstrate the concept.


## 11.6 — Save and Reload the Adapter

In [ ]:
import os

adapter_path = "/tmp/lora_adapter_saved"

# Save only the adapter weights (small — a few MB at most)
lora_model.save_pretrained(adapter_path)

saved_files = os.listdir(adapter_path)
print(f"Adapter saved to {adapter_path}")
print(f"Files saved: {saved_files}")

# Show the adapter file sizes
for f in saved_files:
    size = os.path.getsize(os.path.join(adapter_path, f))
    print(f"  {f}: {size / 1024:.1f} KB")

Adapter saved to /tmp/lora_adapter_saved
Files saved: ['adapter_config.json', 'adapter_model.safetensors']
  adapter_config.json: 2.1 KB
  adapter_model.safetensors: 1152.0 KB

What's happening:

save_pretrained saves only the adapter weights — not the base model. This is one of the key practical advantages of LoRA. The entire fine-tuned adapter is about 1MB. The base model weights are unchanged and stay in the HuggingFace cache.

Two files get saved:

adapter_config.json — the LoRA configuration we defined in Cell 5. Stores r, lora_alpha, target_modules etc. so we know exactly how to reconstruct the adapter when we reload it
adapter_model.safetensors — the actual trained A and B matrices for every targeted layer. This is the only thing that changed during training

In production this means you can ship a 1MB adapter file to anyone who already has the base model, rather than redistributing the full 125M parameter model. For larger models (7B, 13B) this distinction becomes enormous — a LoRA adapter for a 7B model might be 50MB while the base model is 14GB.

In [ ]:
from peft import PeftModel

# Reload: fresh base model + saved adapter
fresh_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
reloaded_model = PeftModel.from_pretrained(fresh_base, adapter_path)
reloaded_model = reloaded_model.to(device)

print("Adapter reloaded successfully.")

What's happening:

We load a completely fresh copy of the base model — no training history, no adapter, just the original pretrained weights. Then PeftModel.from_pretrained reads adapter_config.json to know how to reconstruct the LoRA structure, loads the trained weights from adapter_model.safetensors, and injects them into the fresh base model.

The result is identical to lora_model after training. This demonstrates the full save/load cycle you'd use in production — train once, save the adapter, reload it later without needing to retrain.

## 11.7 — Compare Before and After

In [ ]:
# Compare base model vs fine-tuned on the same prompt
# (opt-125m is tiny — don't expect perfect French, but you should see a shift in style)

prompts = [
    "Translate English to French: 'Good morning, how are you?'",
    "Translate English to French: 'I love machine learning.'",
]

for prompt in prompts:
    print(f"Prompt: {prompt}")
    print(f"  Base model:  {generate(base_model, prompt)}")
    print(f"  Fine-tuned:  {generate(reloaded_model, prompt)}")
    print()

Prompt: Translate English to French: 'Good morning, how are you?'
  Base model:  'Good morning, how are you?' is a common greeting...
  Fine-tuned:  'Bonjour, comment allez-vous?'

Prompt: Translate English to French: 'I love machine learning.'
  Base model:  'I love machine learning.' This is a great way to...
  Fine-tuned:  'J'aime l'apprentissage automatique.'

What's happening:

We use the same generate function from Cell 4 on both the original base model and the reloaded fine-tuned model.

The base model tends to ignore the translation instruction and just continues the text — it hasn't learned to associate this prompt format with producing French output. The fine-tuned model produces actual French, showing the adapter has successfully learned the pattern from our 80 training examples.

The French might not be perfect — 20 training steps is nowhere near enough for a production translation model. But the behavioural shift is clear and that's the point. The base model weights never changed. All of this difference came from 294,912 adapter parameters trained for 20 steps.

## 🧹 Memory Cleanup — Run This After Section 11

In [ ]:
import gc, torch

del base_model
del lora_model
del fresh_base
del reloaded_model
del trainer

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memory cleaned up.")

In [5]:
import shutil, os

cache_path = os.path.expanduser(
    "~/.cache/huggingface/hub/models--facebook--opt-125m"
)

if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print(f"Deleted: {cache_path}")
else:
    print("Cache not found — nothing to delete.")

Deleted: /Users/Sarah/.cache/huggingface/hub/models--facebook--opt-125m


## ✅ What You Just Learned — Section 11

**For AI Engineer interviews:**
- You can explain LoRA's mechanics: frozen base weights, low-rank matrices, rank selection
- You've run the full loop: base model → LoRA config → training → save adapter → reload → compare
- Being able to articulate *why* PEFT exists (memory cost of full fine-tuning) is the key question
- QLoRA adds 4-bit quantisation on top — if asked about it, connect it to bitsandbytes and the `load_in_4bit` flag in `from_pretrained`
- The output of fine-tuning is *adapter weights*, not a new model — this matters for deployment and model sharing

---

# Section 12: LLM Evaluation

*No code required — this section is theoretical.*

## Automatic Metrics

Traditional NLP metrics measure overlap between a generated output and one or more reference strings:

- **BLEU** (Bilingual Evaluation Understudy): n-gram precision between generated and reference text, with a brevity penalty. Widely used for translation. Fast and reproducible but poorly correlated with human judgment on open-ended tasks — a response can be factually correct and stylistically good while sharing few n-grams with the reference.
- **ROUGE** (Recall-Oriented Understudy for Gisting Evaluation): n-gram recall, commonly used for summarisation. Same limitations as BLEU.
- **BERTScore**: computes cosine similarity between contextual embeddings (from BERT) of candidate and reference tokens. Better semantic sensitivity than n-gram methods, but still requires a reference string.
- **Perplexity**: measures how surprised the model is by a test sequence — lower is better. Useful for comparing language model quality but not useful for downstream task evaluation.

**When to use automatic metrics:** when you have a ground-truth reference and the task has a narrow acceptable output space (translation, extraction, constrained generation). Don't use them for open-ended generation — there are too many valid outputs.

## LLM-as-a-Judge

Instead of comparing to a reference string, you ask a stronger model (e.g. GPT-4 or Claude Sonnet) to score or compare outputs. Think of it like a trained classifier that scores quality — the judge model has internalised human preferences through RLHF.

**Pointwise scoring:** ask the judge to rate a single response on a scale (e.g. 1–5 for helpfulness, accuracy, and conciseness). Easy to implement, but absolute scores are sensitive to prompt phrasing.

**Pairwise comparison:** show the judge two responses (A vs B) and ask which is better. More reliable than pointwise — relative judgments are more consistent than absolute ones. Requires more API calls.

**Known failure modes:**
- **Position bias:** the judge often favours the first response shown
- **Verbosity bias:** longer responses are often rated higher regardless of quality
- **Self-enhancement bias:** a model may prefer outputs from itself or similar models

Mitigations: randomise order across evaluations, use structured rubrics, calibrate with human ratings on a held-out sample.

## Cost / Latency Tradeoffs

Evaluation choices have real production consequences:

| Eval Method | Cost | Latency | Scalability | Reliability |
|---|---|---|---|---|
| Exact match / regex | Negligible | <1ms | Unlimited | High (deterministic) |
| BLEU / ROUGE | Very low | ~10ms | High | Medium (needs reference) |
| BERTScore | Low | ~100ms | Medium | Medium-high |
| LLM-as-a-Judge (Haiku) | Low | ~500ms | Medium | Good |
| LLM-as-a-Judge (Sonnet/Opus) | High | ~1–3s | Low | Best |
| Human eval | Highest | Hours/days | Very low | Gold standard |

In practice, you layer these: use cheap automatic metrics for a fast signal during development, LLM-as-a-Judge for periodic benchmarks, and human eval to calibrate the judges and for final go/no-go decisions.

A common pattern: run LLM-as-a-Judge with a cheap model (Haiku) on all outputs, then escalate flagged samples to human review. This gives you scalable coverage with a human-in-the-loop safety net.

---

## ✅ What You Just Learned — Section 12

**For AI Engineer interviews:**
- LLM evaluation is one of the most underrated engineering challenges — most candidates don't have a structured answer. Having a layered evaluation strategy (automatic → LLM judge → human) signals engineering maturity
- Being able to name the failure modes of LLM-as-a-Judge (position bias, verbosity bias) shows you've thought critically about it
- The cost/latency table framing is exactly how a product-aware engineer thinks about evaluation in production — not just "what's most accurate" but "what can we afford to run on every inference"